# BhashaVerse / OneNMT: four-stage corpus-quality evaluation

**Default Stage 1:** untouched checkpoint, full development/test generation and scores, plots and a standalone ZIP. Native BhashaVerse assets require the original SentencePiece model, fairseq dictionary, and verified HF MBart-compatible model. The upstream HF conversion must be checked against independently produced references before research claims. See `docs/four-stage-workflow.md`.


## 1. Experiment settings — only editable cell


In [ ]:
# ONE EDITABLE CELL. Use a fresh notebook session for each stage.
EXPERIMENT = {
    # STAGES: lowest_baseline (default) | low_baseline | decoding_search | language_extension
    "stage": "lowest_baseline",
    "run_label": "corpus-study-1",

    # DATA. Two aligned .txt files; or tsv/jsonl/pre_split.
    "data_mode": "aligned_text",
    "source_lang": "mai_Deva",
    "target_lang": "hin_Deva",
    "source_text": "/kaggle/input/YOUR_DATASET/mai.txt",
    "target_text": "/kaggle/input/YOUR_DATASET/hin.txt",
    "tsv_file": "/kaggle/input/YOUR_DATASET/parallel.tsv",
    "tsv_has_header": False,
    "parallel_jsonl": "data/local/mai-hin/parallel.jsonl",
    "split_directory": "data/local/mai-hin/splits",
    "dataset_id": "corpus-v1",
    "split_seed": 42, "split_ratios": (0.80, 0.10, 0.10),
    "group_by_document": True,
    "validate_data": False,
    "reject_within_split_duplicates": False,  # Allow repeated corpus pairs; ALWAYS block cross-split leakage.
    "data_report": False,

    # MODEL. Must supply compatible local HF MBart safetensors AND original
    # SentencePiece/fairseq dictionary. Verify parity with upstream generation.
    "model_assets": "data/local/onenmt-original",
    "conditioning": "text_prefix",  # Direction marker from original FairseqCodec.
    "beams": 1, "length_penalty": 1.0,   # Untouched baseline uses conservative greedy.
    "max_new_tokens": 64, "max_length": 128, "batch_size": 1,

    # STAGE 2/4 only: real full finetuning.
    "allow_training": False,
    "finetune_mode": "full", "train_seed": 42,
    "epochs": 3, "max_steps": -1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 2e-5, "precision": "fp16",
    "gradient_checkpointing": True, "lora_rank": 8,

    # STAGE 3: dev-only beam search before test.
    "search_source_stage": "low_baseline",
    "search_metric": "chrfpp",
    "search_grid": [
        {"beams": 1, "length_penalty": 1.0},
        {"beams": 4, "length_penalty": 1.0},
    ],

    # STAGE 4 requires separately reviewed repaired dictionary/SPM/embeddings.
    # It does NOT automatically rewrite token IDs or substitute native support.
    "check_assets": True,
    "result_plots": True, "save_archive": True,
    "include_checkpoints": False,
    "setup_mode": "train,repairs",
    "refresh_repo": False,

    # OPTIONAL CLEANUP. Leave "off" normally. Run ONLY settings+cleanup cells to use.
    "cleanup_action": "off",        # off | preview | delete
    "cleanup_target_stage": "low_baseline", # Stage to clean, even if currently in another stage.
    "cleanup_confirmation": "",    # Exact phrase printed by preview; required for delete.
}


## 2. Environment

Choose GPU; the required local model assets must be available.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = (Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir() else Path.cwd().resolve())
os.environ["USE_TF"], os.environ["USE_FLAX"] = "0", "0"
if not (ROOT / "pyproject.toml").is_file():
    if ROOT.exists(): raise ValueError(f"Checkout exists but is incomplete: {ROOT}")
    subprocess.run(["git", "clone", "https://github.com/bhaashik/BhaashikMTFinetuning.git",
                    str(ROOT)], check=True)
elif EXPERIMENT["refresh_repo"]:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
if any(m in sys.modules for m in ("torch","transformers","peft","datasets")):
    raise RuntimeError("Restart session before installing model packages.")
cmd = [sys.executable, str(ROOT / "scripts" / "setup_notebook.py"),
       "--extras", EXPERIMENT["setup_mode"]]
if EXPERIMENT["setup_mode"].startswith("train"):
    cmd.append("--require-cuda")
subprocess.run(cmd, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.experiment_workflow import check_settings
check_settings(EXPERIMENT, model_family="bhashaverse")
print("Checkout:", ROOT)
# Use an independently reviewed HF MBart-compatible BhashaVerse checkpoint.
# Do not substitute generic AutoTokenizer; the Fairseq dictionary IDs matter.


## 3. Convert/split parallel corpus


In [ ]:
from bhaashik.experiment_workflow import (
    prepare_parallel_data, experiment_paths, training_parameters,
    write_experiment_config, resolve_path, report_data_if_enabled,
)
PREPARED = prepare_parallel_data(ROOT, EXPERIMENT, model_family="bhashaverse")
PATHS = experiment_paths(ROOT, EXPERIMENT, family="bhashaverse", prepared=PREPARED)
STAGE_DIR = PATHS["stage"]
ASSETS = resolve_path(ROOT, EXPERIMENT["model_assets"])
report_data_if_enabled(EXPERIMENT, PREPARED, PATHS["report"])
TRAIN_CONFIG = training_parameters(EXPERIMENT, PREPARED, PATHS,
                                   model_family="bhashaverse")
TRAIN_CONFIG_PATH = write_experiment_config(
    EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG)
print("Stage:", EXPERIMENT["stage"], "Assets:", ASSETS, "Outputs:", STAGE_DIR)


## 4. Stage description


In [ ]:
print("BhashaVerse stage:", EXPERIMENT["stage"])
print("Stage 1 = original-model full DEV and TEST predictions; no training.")
print("Stage 2/4 = full training + development loss, then dev/test inference.")
print("Stage 3 = decoding settings selected on dev only.")


## 5. Check local tokenizer/weights


In [ ]:
from bhaashik.staged_experiments import check_model_access
# Reads original dictionary and HF safetensors; no large inference.
check_model_access(EXPERIMENT, family="bhashaverse", root=ROOT)
print("Checkpoint structure checked. Independent upstream output parity must")
print("also be established before making publication claims.")


## 6. Run current stage

Full splits; archiving works without any finetuning.


In [ ]:
from bhaashik.staged_experiments import run_stage
# Validated FairseqCodec only; do not load BhashaVerse via generic AutoTokenizer.
# Every stage produces independent predictions, score reports and a ZIP.
RESULT = run_stage(ROOT, EXPERIMENT, PREPARED, PATHS, TRAIN_CONFIG_PATH,
                   family="bhashaverse")
print("Complete:", RESULT["stage"])


## 7. Review stage scores


In [ ]:
import json
if (STAGE_DIR / "stage_results.json").is_file():
    results = json.loads((STAGE_DIR / "stage_results.json").read_text())
    print("DEV:", results["dev_scores"])
    print("TEST:", results["test_scores"])
    print("Predictions:", results["predictions"])
else:
    print("See stage_manifest.json and archived partial outputs.")


## 8. Archive

The runner writes it automatically even for partial execution.


In [ ]:
print("Stage:", STAGE_DIR)
if PATHS["archive"].is_file():
    print("Preserve archive:", PATHS["archive"])
else:
    print("No stage ZIP; check save_archive or stage_manifest.json.")


## Optional cleanup of a mistaken experiment

**Run this cell separately, not as part of the normal experiment.** In the
first settings cell set `cleanup_action="preview"` and
`cleanup_target_stage` to the mistaken stage. Run only this cleanup
cell to see its files. For deletion, change `cleanup_action="delete"`
and paste the exact printed phrase into `cleanup_confirmation`, then
run this cell alone. It cannot delete source data, split manifests,
other stages, or shared report folders. Default is off.


In [ ]:
# OPTIONAL: clean an incorrect generated experiment, without touching source data.
# To clean: set cleanup_action="preview" in the FIRST settings cell and run
# this cell ONLY (after the settings cell); read the files it lists.
# To delete: set cleanup_action="delete", copy its exact confirmation phrase
# into cleanup_confirmation, and run THIS CELL ONLY. Do not click Run All.
# The selected stage, its ZIP and stage-specific configs are the only targets.
from pathlib import Path
from bhaashik.experiment_cleanup import cleanup_stage

if EXPERIMENT["cleanup_action"] == "off":
    print("Cleanup disabled. To use it, edit cleanup_* in the first settings cell.")
else:
    root_for_cleanup = (
        Path("/kaggle/working/BhaashikMTFinetuning")
        if Path("/kaggle/working").is_dir()
        else Path("/content/BhaashikMTFinetuning")
        if Path("/content").is_dir()
        else Path.cwd()
    )
    cleanup_plan = cleanup_stage(
        root_for_cleanup, EXPERIMENT,
        family="bhashaverse",
        stage=EXPERIMENT["cleanup_target_stage"],
        action=EXPERIMENT["cleanup_action"],
        confirmation=EXPERIMENT["cleanup_confirmation"],
    )
    print("Deletion completed:", cleanup_plan["deleted"])
